In [7]:
# python
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import funciones as fn

In [8]:
# Rutas y nombre de la columna objetivo
train_path = "dataset/train.csv"
val_path = "dataset/val.csv"
target_column = "shares"

# Cargar datos
df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

In [9]:
# Columnas especificadas manualmente vía funciones
numeric_cols = fn.columnas_numericas()
#numeric_cols = fn.columnas_cantidad()
#ohe_cols = fn.columnas_one_hot()
ohe_cols = fn.columnas_one_hot_dia_semana() + fn.columnas_one_hot_tematica()
use_imputer_for_ohe = False  # True si las OHE tienen NaNs

In [10]:

# Validar existencia de columnas
missing = [c for c in numeric_cols + ohe_cols + [target_column] if c not in df_train.columns]
if missing:
    raise ValueError(f"Faltan columnas en `train`: {missing}")
missing = [c for c in numeric_cols + ohe_cols + [target_column] if c not in df_val.columns]
if missing:
    raise ValueError(f"Faltan columnas en `val`: {missing}")

In [11]:

# Separar X / y
X_train = df_train.drop(columns=[target_column])
y_train = df_train[target_column]
X_val = df_val.drop(columns=[target_column])
y_val = df_val[target_column]

In [12]:
# Construir transformadores
transformers = []
if numeric_cols:
    num_pipeline = Pipeline([('scaler', StandardScaler())])
    transformers.append(('num', num_pipeline, numeric_cols))

if ohe_cols:
    transformers.append(('ohe_given', 'passthrough', ohe_cols))

if not transformers:
    raise ValueError('No se definieron columnas numéricas ni OHE.')

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

# Modelo Random Forest con parámetros fijos
rf = RandomForestRegressor(
    n_estimators=2000,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=4,
    max_features='log2',
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

# Pipeline con el preprocesamiento + modelo
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', rf)
])

# Entrenamiento directo
pipeline.fit(X_train, y_train)

# El modelo entrenado es el pipeline
best_model = pipeline

# Función de evaluación usando el mejor modelo
def evaluar(nombre, X, y_true):
    y_pred = best_model.predict(X)  # revertimos log1p
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"--- {nombre} ---")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R2: {r2:.4f}")

# Evaluar en validación
evaluar('validation', X_val, y_val)

--- validation ---
MSE: 1057957.3035
RMSE: 1028.5705
MAE: 768.5760
R2: 0.1062


Conclusion:
- Error medio absoluto alto, se equivoca en promedio casi 750 shares
- R2 bajo, el modelo captura a penas 10% de la relacion entre las variable, predice casi como la media